In [ ]:
# Install required packages
import sys
import subprocess
for package in ['tensorflow', 'opencv-python', 'streamlit']:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

## Load and Preprocess Crop Image Dataset

In [ ]:
from pathlib import Path
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split

ROOT = Path('archive')
SUNFLOWER_ROOT = Path('Sunflower Compressed')
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
TARGET_SIZE = (128, 128)


def all_images(folder):
    return [p for p in folder.rglob('*') if p.suffix.lower() in IMG_EXTS]


def class_images(root):
    result = {}
    for d in sorted(root.iterdir()):
        if d.is_dir():
            imgs = all_images(d)
            if imgs:
                result[d.name] = imgs
    return result

cucumber = class_images(ROOT)
sunflower = class_images(SUNFLOWER_ROOT)

images = []
labels = []
for cls, paths in list(cucumber.items()) + list(sunflower.items()):
    crop = 'cucumber' if cls in cucumber else 'sunflower'
    for p in paths:
        images.append(img_to_array(load_img(p, target_size=TARGET_SIZE)))
        labels.append((crop, cls, 'healthy' if 'healthy' in cls.lower() else 'unhealthy'))

images = np.array(images, dtype='float32') / 255.0
crop_types = sorted({x[0] for x in labels})
stages = sorted({x[1] for x in labels})
healths = sorted({x[2] for x in labels})

crop_type_map = {v: i for i, v in enumerate(crop_types)}
stage_map = {v: i for i, v in enumerate(stages)}
health_map = {v: i for i, v in enumerate(healths)}

y_type = np.array([crop_type_map[x[0]] for x in labels])
y_stage = np.array([stage_map[x[1]] for x in labels])
y_health = np.array([health_map[x[2]] for x in labels])

X_train, X_val, t_train, t_val, s_train, s_val, h_train, h_val = train_test_split(
    images, y_type, y_stage, y_health, test_size=0.2, random_state=42, stratify=y_type
)

print('Train samples', len(X_train), 'Validation samples', len(X_val))
print('Crop types', crop_types)
print('Stages', stages)
print('Health labels', healths)

## Define Lightweight CNN Model

In [ ]:
from tensorflow.keras import layers, models

input_shape = TARGET_SIZE + (3,)
num_type = len(crop_types)
num_stage = len(stages)
num_health = len(healths)

inputs = layers.Input(shape=input_shape)
x = layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation='relu')(x)

out_type = layers.Dense(num_type, activation='softmax', name='type')(x)
out_stage = layers.Dense(num_stage, activation='softmax', name='stage')(x)
out_health = layers.Dense(num_health, activation='softmax', name='health')(x)

model = models.Model(inputs, [out_type, out_stage, out_health])
model.compile(
    optimizer='adam',
    loss={'type': 'sparse_categorical_crossentropy', 'stage': 'sparse_categorical_crossentropy', 'health': 'sparse_categorical_crossentropy'},
    metrics={'type': 'accuracy', 'stage': 'accuracy', 'health': 'accuracy'}
)

model.summary()

y_train = {'type': t_train, 'stage': s_train, 'health': h_train}
y_val = {'type': t_val, 'stage': s_val, 'health': h_val}

## Train Model for Crop Type, Stage, and Health

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=16,
    verbose=1
)
model.save('crop_classification_model')

## Compute Explanations with Grad-CAM

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

last_conv_layer = model.layers[-6]


def make_gradcam_heatmap(img, model, class_index, output_name):
    grad_model = tf.keras.models.Model(
        [model.inputs], [last_conv_layer.output, model.get_layer(output_name).output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(np.expand_dims(img, axis=0))
        loss = predictions[:, class_index]
    grads = tape.gradient(loss, conv_outputs)[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs[0]), axis=-1)
    heatmap = np.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return np.uint8(255 * heatmap.numpy())

sample_img = X_val[0]
preds = model.predict(np.expand_dims(sample_img, axis=0))
for name, pred, labels_map in [('type', preds[0][0], crop_types), ('stage', preds[1][0], stages), ('health', preds[2][0], healths)]:
    idx = int(np.argmax(pred))
    heatmap = make_gradcam_heatmap(sample_img, model, idx, name)
    plt.imshow(heatmap, cmap='jet')
    plt.title(f'{name}: {labels_map[idx]}')
    plt.axis('off')
    plt.show()

## Evaluate Model Performance

In [ ]:
results = model.evaluate(X_val, y_val, verbose=1)
print('Validation results:', results)

## Create Streamlit App for Prediction and Explanation

In [ ]:
streamlit_code = '''import streamlit as st
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array

model = load_model('crop_classification_model')

crop_types = %s
stages = %s
healths = %s

def make_gradcam_heatmap(img, class_index, output_name):
    last_conv_layer = model.layers[-6]
    grad_model = tf.keras.models.Model(
        [model.inputs], [last_conv_layer.output, model.get_layer(output_name).output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(np.expand_dims(img, axis=0))
        loss = predictions[:, class_index]
    grads = tape.gradient(loss, conv_outputs)[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs[0]), axis=-1)
    heatmap = np.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return np.uint8(255 * heatmap.numpy())

st.title('Crop Classification')
img_file = st.file_uploader('Upload image', type=['jpg', 'jpeg', 'png', 'bmp', 'webp'])
if img_file:
    img = Image.open(img_file).convert('RGB').resize((128, 128))
    st.image(img, caption='Uploaded image', use_column_width=True)
    x = img_to_array(img).astype('float32') / 255.0
    preds = model.predict(np.expand_dims(x, axis=0))
    type_idx = int(np.argmax(preds[0][0]))
    stage_idx = int(np.argmax(preds[1][0]))
    health_idx = int(np.argmax(preds[2][0]))
    st.write('Crop type:', crop_types[type_idx])
    st.write('Stage:', stages[stage_idx])
    st.write('Health:', healths[health_idx])
    heatmap = make_gradcam_heatmap(x, type_idx, 'type')
    st.image(heatmap, caption='Grad-CAM heatmap', use_column_width=True)
'''
with open('streamlit_app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code % (crop_types, stages, healths))
print('Saved streamlit_app.py')


In [ ]:
from pathlib import Path
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split

ROOT = Path('archive')
SUNFLOWER_ROOT = Path('Sunflower Compressed')
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
TARGET_SIZE = (128, 128)


def all_images(folder):
    return [p for p in folder.rglob('*') if p.suffix.lower() in IMG_EXTS]


def class_images(root):
    result = {}
    for d in sorted(root.iterdir()):
        if d.is_dir():
            imgs = all_images(d)
            if imgs:
                result[d.name] = imgs
    return result

cucumber = class_images(ROOT)
sunflower = class_images(SUNFLOWER_ROOT)

images = []
labels = []
for crop, items in [('cucumber', cucumber), ('sunflower', sunflower)]:
    for cls, paths in items.items():
        for p in paths:
            images.append(img_to_array(load_img(p, target_size=TARGET_SIZE)))
            health = 'healthy' if 'healthy' in cls.lower() else 'unhealthy' if any(x in cls.lower() for x in ['unhealthy', 'wilted']) else 'healthy'
            labels.append((crop, cls, health))

images = np.array(images, dtype='float32') / 255.0
crop_types = sorted({x[0] for x in labels})
stages = sorted({x[1] for x in labels})
healths = sorted({x[2] for x in labels})

crop_type_map = {v: i for i, v in enumerate(crop_types)}
stage_map = {v: i for i, v in enumerate(stages)}
health_map = {v: i for i, v in enumerate(healths)}

y_type = np.array([crop_type_map[x[0]] for x in labels])
y_stage = np.array([stage_map[x[1]] for x in labels])
y_health = np.array([health_map[x[2]] for x in labels])

X_train, X_val, t_train, t_val, s_train, s_val, h_train, h_val = train_test_split(
    images, y_type, y_stage, y_health, test_size=0.2, random_state=42, stratify=y_type
)

print('Train samples', len(X_train), 'Validation samples', len(X_val))
print('Crop types', crop_types)
print('Stages', stages)
print('Health labels', healths)

## Define Lightweight CNN Model

In [ ]:
from tensorflow.keras import layers, models

input_shape = TARGET_SIZE + (3,)
num_type = len(crop_types)
num_stage = len(stages)
num_health = len(healths)

inputs = layers.Input(shape=input_shape)
x = layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation='relu')(x)

out_type = layers.Dense(num_type, activation='softmax', name='type')(x)
out_stage = layers.Dense(num_stage, activation='softmax', name='stage')(x)
out_health = layers.Dense(num_health, activation='softmax', name='health')(x)

model = models.Model(inputs, [out_type, out_stage, out_health])
model.compile(
    optimizer='adam',
    loss={'type': 'sparse_categorical_crossentropy', 'stage': 'sparse_categorical_crossentropy', 'health': 'sparse_categorical_crossentropy'},
    metrics={'type': 'accuracy', 'stage': 'accuracy', 'health': 'accuracy'}
)

model.summary()

y_train = {'type': t_train, 'stage': s_train, 'health': h_train}
y_val = {'type': t_val, 'stage': s_val, 'health': h_val}

## Train Model for Crop Type, Stage, and Health

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=16,
    verbose=1
)
model.save('crop_classification_model')

## Compute Explanations with Grad-CAM

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

last_conv_layer = model.layers[-6]


def make_gradcam_heatmap(img, model, class_index, output_name):
    grad_model = tf.keras.models.Model(
        [model.inputs], [last_conv_layer.output, model.get_layer(output_name).output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(np.expand_dims(img, axis=0))
        loss = predictions[:, class_index]
    grads = tape.gradient(loss, conv_outputs)[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs[0]), axis=-1)
    heatmap = np.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return np.uint8(255 * heatmap.numpy())

sample_img = X_val[0]
preds = model.predict(np.expand_dims(sample_img, axis=0))
for name, pred, labels_map in [('type', preds[0][0], crop_types), ('stage', preds[1][0], stages), ('health', preds[2][0], healths)]:
    idx = int(np.argmax(pred))
    heatmap = make_gradcam_heatmap(sample_img, model, idx, name)
    plt.imshow(heatmap, cmap='jet')
    plt.title(f'{name}: {labels_map[idx]}')
    plt.axis('off')
    plt.show()

## Evaluate Model Performance

In [ ]:
results = model.evaluate(X_val, y_val, verbose=1)
print('Validation results:', results)

## Create Streamlit App for Prediction and Explanation

In [ ]:
streamlit_code = '''import streamlit as st
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array

model = load_model('crop_classification_model')

crop_types = %s
stages = %s
healths = %s

def make_gradcam_heatmap(img, class_index, output_name):
    last_conv_layer = model.layers[-6]
    grad_model = tf.keras.models.Model(
        [model.inputs], [last_conv_layer.output, model.get_layer(output_name).output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(np.expand_dims(img, axis=0))
        loss = predictions[:, class_index]
    grads = tape.gradient(loss, conv_outputs)[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs[0]), axis=-1)
    heatmap = np.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return np.uint8(255 * heatmap.numpy())

st.title('Crop Classification')
img_file = st.file_uploader('Upload image', type=['jpg', 'jpeg', 'png', 'bmp', 'webp'])
if img_file:
    img = Image.open(img_file).convert('RGB').resize((128, 128))
    st.image(img, caption='Uploaded image', use_column_width=True)
    x = img_to_array(img).astype('float32') / 255.0
    preds = model.predict(np.expand_dims(x, axis=0))
    type_idx = int(np.argmax(preds[0][0]))
    stage_idx = int(np.argmax(preds[1][0]))
    health_idx = int(np.argmax(preds[2][0]))
    st.write('Crop type:', crop_types[type_idx])
    st.write('Stage:', stages[stage_idx])
    st.write('Health:', healths[health_idx])
    heatmap = make_gradcam_heatmap(x, type_idx, 'type')
    st.image(heatmap, caption='Grad-CAM heatmap', use_column_width=True)
'''
with open('streamlit_app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code % (crop_types, stages, healths))
print('Saved streamlit_app.py')


# Crop Classification Notebook